# Xiaoyang Slides — 个人部分可视化（一键导出）

为 slide 里**你自己负责的章节**（IMU Expert + Phase Arbitrator）生成 5 张图：

| 图 | 用途 | 章节 |
|---|---|---|
| **fig1_accuracy_overview** | 4 个 baseline 准确率柱状图 | Headline / context |
| **fig2_phase_features_multi** | 4 个动作的物理特征对比 ⭐ | Phase Arbitrator（核心 novelty 图）|
| **fig3_alpha_untrained** | 多动作未训练 α(t) | Phase Arbitrator baseline |
| **fig4_imu_per_class** | IMU 单模态 27 类准确率排序 | IMU Expert / Failure Analysis |
| **fig5_imu_confusion** | IMU 混淆矩阵 | IMU Expert |

从上往下跑，跑完浏览器自动下载 zip。需要的文件：
- `MMAI/utd_mhad/Inertial/*.mat`
- `MMAI/pgmoe_ckpt/imu_classifier_best.pt`

## 1. 挂 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. 路径 + 输出目录

In [ ]:
import os

MMAI       = "/content/drive/MyDrive/MMAI"
DATA_ROOT  = f"{MMAI}/utd_mhad"
INERTIAL   = f"{DATA_ROOT}/Inertial"

# 找 IMU 权重 (可能在 MMAI/pgmoe_ckpt/ 或 MMAI/Models/)
def find_first(name, dirs):
    for d in dirs:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    return None

IMU_CLF = find_first("imu_classifier_best.pt",
                     [f"{MMAI}/pgmoe_ckpt", f"{MMAI}/Models"])

# 输出目录: 优先 repo figures, 否则 MMAI
REPO_FIG = "/content/drive/MyDrive/Multi-Modal-AI/project/final/figures/xiaoyang_slides"
SAVE_DIR = REPO_FIG if os.path.exists(os.path.dirname(REPO_FIG)) else f"{MMAI}/figures/xiaoyang_slides"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"IMU_CLF  : {IMU_CLF}")
print(f"SAVE_DIR : {SAVE_DIR}")
assert IMU_CLF, "imu_classifier_best.pt 没找到"
assert os.path.exists(INERTIAL), f"Inertial 文件夹没找到: {INERTIAL}"

## 3. Imports + 工具函数

In [ ]:
import time
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMU_LEN = 192
UTD_LABELS = [
    "swipe left", "swipe right", "wave", "clap", "throw",
    "arm cross", "basketball shoot", "draw x",
    "draw circle CW", "draw circle CCW", "draw triangle",
    "bowling", "boxing", "baseball swing", "tennis swing",
    "arm curl", "tennis serve", "two hand push", "knock",
    "catch", "pickup and throw", "jogging", "walking",
    "sit to stand", "stand to sit", "forward lunge", "squat",
]

def load_imu(action, subject=1, trial=1):
    fpath = f"{INERTIAL}/a{action}_s{subject}_t{trial}_inertial.mat"
    if not os.path.exists(fpath):
        return None
    data = sio.loadmat(fpath)["d_iner"].astype(np.float32)
    if data.shape[0] < IMU_LEN:
        data = np.concatenate([data, np.zeros((IMU_LEN - data.shape[0], 6), np.float32)], axis=0)
    return torch.from_numpy(data[:IMU_LEN]).T.contiguous()  # (6, 192)

## 4. 定义模型类（inline，让 notebook self-contained）

In [ ]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_c, out_c, kernel=5, stride=1):
        super().__init__()
        pad = kernel // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel, stride=stride, padding=pad),
            nn.BatchNorm1d(out_c), nn.ReLU(inplace=True),
            nn.Conv1d(out_c, out_c, kernel, stride=1, padding=pad),
            nn.BatchNorm1d(out_c),
        )
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv1d(in_c, out_c, 1, stride=stride), nn.BatchNorm1d(out_c))
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))

class IMUExpert(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(6, 64, 7, stride=2, padding=3), nn.BatchNorm1d(64), nn.ReLU(inplace=True))
        self.block1 = ResidualBlock1D(64, 128, 5, 2)
        self.block2 = ResidualBlock1D(128, 256, 5, 2)
        self.block3 = ResidualBlock1D(256, d_model, 3, 2)
    def forward(self, x):
        x = self.stem(x); x = self.block1(x); x = self.block2(x); x = self.block3(x)
        return x.transpose(1, 2)

class IMUClassifier(nn.Module):
    def __init__(self, num_classes=27, d_model=256):
        super().__init__()
        self.encoder = IMUExpert(d_model=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        tokens = self.encoder(x)
        return self.head(self.norm(tokens.mean(dim=1)))

class PhaseFeatures(nn.Module):
    def forward(self, x):
        acc = x[:, :3]
        mag = torch.sqrt((acc ** 2).sum(dim=1, keepdim=True) + 1e-8)
        mag_d = torch.diff(mag, dim=2, prepend=mag[:, :, :1])
        mag_dd = torch.diff(mag_d, dim=2, prepend=mag_d[:, :, :1])
        energy = (acc ** 2).sum(dim=1, keepdim=True)
        e_rate = torch.diff(energy, dim=2, prepend=energy[:, :, :1])
        return torch.cat([mag, mag_dd, e_rate], dim=1)

class PhaseArbitrator(nn.Module):
    def __init__(self, T_i=12):
        super().__init__()
        self.features = PhaseFeatures()
        self.pool = nn.AdaptiveAvgPool1d(T_i)
        self.encoder = nn.Sequential(
            nn.Conv1d(3, 64, 1), nn.ReLU(inplace=True),
            nn.Conv1d(64, 32, 1), nn.ReLU(inplace=True),
        )
        self.arb = nn.Sequential(
            nn.Conv1d(32, 16, 1), nn.ReLU(inplace=True),
            nn.Conv1d(16, 1, 1), nn.Sigmoid(),
        )
    def forward(self, imu):
        f = self.pool(self.features(imu))
        return self.arb(self.encoder(f)).squeeze(1)

## 5. Fig 1 — 总体准确率对比（headline）

4 个 baseline 并排：midterm IMU 67.9% → IMU expert 82.33% → Vision 89.3% → PG-MoE v1 89.77%。

In [ ]:
methods = ["Midterm IMU\n(1D-CNN)", "IMU Expert\n(deep+res)", "Vision\n(ResNet3D)", "PG-MoE final\n(team)"]
accs    = [67.9, 82.33, 89.3, 91.0]
colors  = ["#9CA3AF", "#3B82F6", "#10B981", "#EF4444"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(methods, accs, color=colors, edgecolor='black', linewidth=1.2, alpha=0.92)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 1, f'{acc}%',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel("Test Accuracy (%)", fontsize=13)
ax.set_title("Accuracy Progression: Single Modality → PG-MoE", fontsize=15, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.25, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out1 = f"{SAVE_DIR}/fig1_accuracy_overview.png"
plt.savefig(out1, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out1}")

## 6. Fig 2 — 多动作物理特征对比 ⭐（核心 novelty 图）

4 个动作的 `|a(t)|` / `d²|a|/dt²` / `energy_rate` 并排：
- **Throw**：impact 型，sharp spike
- **Walking**：周期性，规律起伏
- **Swipe Left**：平滑，单 peak
- **Draw circle CCW**：长持续中等加速

这张图直接论证 phase arbitrator 的输入 features 在不同动作下**信号完全不同**——为 paper method 章节背书。

In [ ]:
phase_extractor = PhaseFeatures()

demo_actions = [
    (5,  "Throw (impact)",       "#EF4444"),
    (23, "Walking (periodic)",   "#3B82F6"),
    (1,  "Swipe L (smooth)",     "#10B981"),
    (10, "Draw circle CCW",      "#8B5CF6"),
]

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
feature_names = [r"$|a(t)|$", r"$d^2|a|/dt^2$", r"energy rate"]

for col, (act, name, color) in enumerate(demo_actions):
    imu = load_imu(act).unsqueeze(0)
    feats = phase_extractor(imu)
    t = np.arange(IMU_LEN)
    for r in range(3):
        axes[r, col].plot(t, feats[0, r].numpy(), color=color, linewidth=1.6)
        axes[r, col].grid(True, alpha=0.2)
        axes[r, col].spines['top'].set_visible(False)
        axes[r, col].spines['right'].set_visible(False)
        if col == 0:
            axes[r, col].set_ylabel(feature_names[r], fontsize=12)
    axes[0, col].set_title(name, fontsize=13, fontweight='bold')
    axes[2, col].set_xlabel("timestep", fontsize=11)

fig.suptitle("Phase Features Differ Sharply Across Action Types\n(input to Phase Arbitrator)",
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
out2 = f"{SAVE_DIR}/fig2_phase_features_multi.png"
plt.savefig(out2, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out2}")

## 7. Fig 3 — α(t) untrained vs trained 对比 ⭐

**左**：随机权重 → α 全贴 0.5（无 phase awareness）
**右**：PG-MoE 联合训练后 → α 收敛到 data-driven 模式

这张对比图证明 **phase awareness 是学出来的，不是 hardcode**。

**前提**：需要 `pgmoe_best.pt` 在 `MMAI/pgmoe_ckpt/`。如果没有，cell 会自动 fallback 到只画 untrained 并提示上传。

In [ ]:
# 1) 算 untrained α (随机权重)
torch.manual_seed(42)
pa_untrained = PhaseArbitrator(T_i=12)

demo_actions = [
    (5,  "Throw (impact)",   "#EF4444"),
    (23, "Walking (periodic)", "#3B82F6"),
    (1,  "Swipe L (smooth)",   "#10B981"),
    (10, "Draw circle CCW",    "#8B5CF6"),
]

untrained = {}
for act, name, color in demo_actions:
    imu = load_imu(act).unsqueeze(0)
    with torch.no_grad():
        untrained[name] = (pa_untrained(imu).squeeze().numpy(), color)

# 2) 尝试加载 PG-MoE 的 phase_arbitrator 权重
trained = {}
pgmoe_candidates = [
    f"{MMAI}/pgmoe_ckpt/pgmoe_best.pt",
    f"{MMAI}/Models/pgmoe_best.pt",
    f"{MMAI}/pgmoe_ckpt/pgmoe_late_best.pt",
]
loaded_from = None
for p in pgmoe_candidates:
    if not os.path.exists(p):
        continue
    try:
        full_sd = torch.load(p, map_location='cpu', weights_only=False)
        if isinstance(full_sd, dict) and 'model_state' in full_sd:
            full_sd = full_sd['model_state']
        pa_keys = {k.replace('phase_arbitrator.', ''): v
                   for k, v in full_sd.items()
                   if k.startswith('phase_arbitrator.')}
        if not pa_keys:
            continue
        pa_trained = PhaseArbitrator(T_i=12)
        pa_trained.load_state_dict(pa_keys)
        pa_trained.eval()
        for act, name, color in demo_actions:
            imu = load_imu(act).unsqueeze(0)
            with torch.no_grad():
                trained[name] = (pa_trained(imu).squeeze().numpy(), color)
        loaded_from = p
        break
    except Exception as e:
        print(f"  skip {p}: {e}")

# 3) 画图: 有 trained 就左右对比, 没有就 fallback 到只画 untrained
if trained:
    print(f"✅ Loaded trained phase_arbitrator from {loaded_from}")
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for name, (a, c) in untrained.items():
        axes[0].plot(np.linspace(0, 1, 12), a, "o-", color=c, label=name, linewidth=2, markersize=7)
    for name, (a, c) in trained.items():
        axes[1].plot(np.linspace(0, 1, 12), a, "o-", color=c, label=name, linewidth=2, markersize=7)
    for ax in axes:
        ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
        ax.set_ylim(0, 1)
        ax.set_xlabel("normalized time", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[0].set_title("Untrained (random init)", fontsize=13, fontweight='bold')
    axes[1].set_title("After PG-MoE joint training", fontsize=13, fontweight='bold')
    axes[0].set_ylabel(r"$\alpha$ (1=trust vision, 0=trust IMU)", fontsize=12)
    fig.suptitle(r"$\alpha(t)$: Phase Awareness Is Learned, Not Hardcoded",
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    out3 = f"{SAVE_DIR}/fig3_alpha_comparison.png"
else:
    print("⚠ 没找到 pgmoe_best.pt — 只画 untrained")
    print(f"   把 pgmoe_archive/pgmoe_best.pt 上传到 {MMAI}/pgmoe_ckpt/ 才能画 trained 对比")
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, (a, c) in untrained.items():
        ax.plot(np.linspace(0, 1, 12), a, "o-", color=c, label=name, linewidth=2, markersize=8)
    ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
    ax.set_ylim(0, 1)
    ax.set_xlabel("normalized time", fontsize=12)
    ax.set_ylabel(r"$\alpha$", fontsize=12)
    ax.set_title(r"$\alpha(t)$ — Untrained Baseline", fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3); ax.legend(loc='best')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    out3 = f"{SAVE_DIR}/fig3_alpha_untrained.png"

plt.savefig(out3, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ {out3}")

## 8. 加载 IMU 分类器 + 跑 test set

下面 3 张图都基于这个评估结果。

In [ ]:
class IMUDataset(Dataset):
    def __init__(self, train=False):
        allowed = {1, 3, 5, 7} if train else {2, 4, 6, 8}
        self.samples = []
        for fn in sorted(os.listdir(INERTIAL)):
            if not fn.endswith("_inertial.mat"): continue
            parts = fn.split("_")
            a, s = int(parts[0][1:]), int(parts[1][1:])
            if s not in allowed: continue
            self.samples.append((os.path.join(INERTIAL, fn), a - 1))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fp, lb = self.samples[idx]
        d = sio.loadmat(fp)["d_iner"].astype(np.float32)
        if d.shape[0] < IMU_LEN:
            d = np.concatenate([d, np.zeros((IMU_LEN - d.shape[0], 6), np.float32)], axis=0)
        return torch.from_numpy(d[:IMU_LEN]).T.contiguous(), lb

test_ds = IMUDataset(train=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

imu_clf = IMUClassifier().to(device)
imu_clf.load_state_dict(torch.load(IMU_CLF, map_location=device, weights_only=False))
imu_clf.eval()

preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.extend(imu_clf(x.to(device)).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
preds, labels = np.array(preds), np.array(labels)
imu_acc = accuracy_score(labels, preds)
print(f"IMU expert test acc: {imu_acc:.4f} ({imu_acc*100:.2f}%)")

imu_per_class = {}
for c in range(27):
    mask = labels == c
    if mask.sum() > 0:
        imu_per_class[c] = (preds[mask] == c).mean()

## 9. Fig 4 — IMU 单模态 per-class 准确率（按 acc 升序）

红=最差 3 类（IMU 失效，需要 vision 接管），黄=中等，绿=已经接近完美。

In [ ]:
order = sorted(imu_per_class.items(), key=lambda r: r[1])

fig, ax = plt.subplots(figsize=(11, 9))
colors_bar = ['#EF4444' if a < 0.5 else '#F59E0B' if a < 0.8 else '#10B981' for _, a in order]
y_pos = np.arange(len(order))
ax.barh(y_pos, [a*100 for _, a in order], color=colors_bar, edgecolor='black', alpha=0.88)

for i, (c, a) in enumerate(order):
    ax.text(a*100 + 1, i, f'{a*100:.1f}%', va='center', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"c{c}: {UTD_LABELS[c]}" for c, _ in order], fontsize=10)
ax.set_xlabel("Accuracy (%)", fontsize=12)
ax.set_xlim(0, 105)
ax.axvline(50, ls=":", color="red",  alpha=0.4)
ax.axvline(80, ls=":", color="gray", alpha=0.4)
ax.set_title(f"IMU Expert: Per-Class Accuracy ({imu_acc*100:.2f}% overall)\n"
             "Worst 3 (red) reveal where Vision should take over",
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
out4 = f"{SAVE_DIR}/fig4_imu_per_class.png"
plt.savefig(out4, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out4}")

## 10. Fig 5 — IMU 混淆矩阵

In [ ]:
cm = confusion_matrix(labels, preds, labels=list(range(27)))

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax, shrink=0.85)

for i in range(27):
    for j in range(27):
        if cm[i, j] > 0:
            color = 'white' if cm[i, j] > cm.max() * 0.55 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=8)

short = [l[:14] for l in UTD_LABELS]
ax.set_xticks(range(27)); ax.set_yticks(range(27))
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short, fontsize=8)
ax.set_xlabel("Predicted class", fontsize=12)
ax.set_ylabel("True class", fontsize=12)
ax.set_title(f"IMU Expert Confusion Matrix  (test acc {imu_acc*100:.2f}%)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
out5 = f"{SAVE_DIR}/fig5_imu_confusion.png"
plt.savefig(out5, dpi=200, bbox_inches='tight'); plt.show()
print(f"✅ {out5}")

## 11. 打包 zip + 下载到本地

In [ ]:
import shutil
zip_path = "/content/xiaoyang_figures.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", SAVE_DIR)

print(f"Bundle contents:")
for fn in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR, fn)) / 1024
    print(f"  {fn}  ({sz:.1f} KB)")
print(f"\nzipped: {os.path.getsize(zip_path)/1024:.1f} KB")

from google.colab import files
files.download(zip_path)

## 完事

zip 下载到本地，解压塞到 slide 里。每张图都已经有 title + 坐标轴 + 图例，直接用。

**slide 用法建议：**

| Slide 章节 | 用哪张图 |
|---|---|
| Headline / Context | fig1 |
| Phase Arbitrator method | **fig2** ⭐, fig3 |
| IMU Expert / Failure Analysis | fig4 + fig5 |